# Step 3 (variant): Distinguishing MISSING pages from BLANK fields

This notebook demonstrates the optional **missing-page handling** feature added in response to [issue #317](https://github.com/aws-solutions-library-samples/accelerated-intelligent-document-processing-on-aws/issues/317).

## Why this matters

Many forms have optional sub-sections (worksheets, supplements) that submitters can omit. With the default extraction behavior, the system can't tell:

- **BLANK** — the page was submitted but the field was unfilled, vs.
- **MISSING** — the page wasn't submitted at all

The LLM doesn't know which pages *should* exist, so prompt-engineering this distinction does not work reliably. The fix moves the page-presence knowledge into config and code.

## What this notebook shows

Using the `samples/bank-statement-multipage.pdf` sample and the saved output from `step2_classification.ipynb`, we will:

1. **Run A (control)** — extract from the full document with the feature disabled. The output contains all schema fields, populated where the model found values.
2. **Run B (simulated omission)** — drop the transactions-worksheet page from the section, enable the new `extraction.missing_field_handling`, and extract again. Fields whose declared source pages are absent now appear in `missing_fields_report` instead of being silently empty.

We then diff the two `inference_result` objects to make the BLANK vs MISSING distinction concrete.

## Prerequisites

- `step1_ocr.ipynb` and `step2_classification.ipynb` have been run (this notebook reads `.data/step2_classification/document.json`).
- AWS credentials with Bedrock access in your shell environment.
- The `idp_common[extraction]` package installed (`pip install -e ../../../lib/idp_common_pkg[extraction]`).

## 1. Load classification output and the config

In [ ]:
import copy
import json
import logging
import os
from pathlib import Path

import boto3
import yaml

from idp_common import extraction
from idp_common.models import Document

logging.basicConfig(level=logging.WARNING)
logging.getLogger('idp_common.extraction').setLevel(logging.INFO)

print('Libraries imported.')

In [ ]:
classification_data_dir = Path('.data/step2_classification')
with open(classification_data_dir / 'document.json') as f:
    document = Document.from_json(f.read())

config_dir = Path('config')
CONFIG = {}
for cfg_file in ['extraction.yaml', 'classes.yaml']:
    with open(config_dir / cfg_file) as f:
        CONFIG.update(yaml.safe_load(f))

with open(classification_data_dir / 'environment.json') as f:
    env_info = json.load(f)
os.environ['AWS_REGION'] = env_info['region']
os.environ['METRIC_NAMESPACE'] = 'IDP-Modular-Pipeline'

section = document.sections[0] if document.sections else None
if section is None:
    raise RuntimeError('No sections found — did step2_classification.ipynb run successfully?')

print(f'Document: {document.id}')
print(f'Section: {section.section_id} ({section.classification})')
print(f'Section pages: {section.page_ids}')

## 2. Add the page-type schema extensions

The original `classes.yaml` has no notion of page sub-types. We declare two on the Bank Statement class — `AccountSummary` (with account-holder and statement-period info) and `TransactionsWorksheet` (with the line-items table) — and annotate each property with the page-type it sources from. The regex patterns reuse the existing `x-aws-idp-document-page-content-regex` extension.

The original `CONFIG` is left untouched; we add the extensions to a copy used only for this notebook.

In [ ]:
augmented = copy.deepcopy(CONFIG)
bank_statement = next(c for c in augmented['classes'] if c.get('$id') == 'Bank Statement')

bank_statement['x-aws-idp-page-types'] = [
    {
        'name': 'AccountSummary',
        'description': 'Page with account holder and statement period info',
        'x-aws-idp-document-page-content-regex': '(?i)(account\\s+summary|statement\\s+period|account\\s+holder)',
    },
    {
        'name': 'TransactionsWorksheet',
        'description': 'Transactions ledger',
        'x-aws-idp-document-page-content-regex': '(?i)(transactions?|deposits?\\s+and\\s+withdrawals|date\\s*\\|\\s*description)',
    },
]

props = bank_statement['properties']
props['Account Holder Address']['x-aws-idp-source-page-types'] = ['AccountSummary']
props['Account Number']['x-aws-idp-source-page-types'] = ['AccountSummary']
props['Statement Period']['x-aws-idp-source-page-types'] = ['AccountSummary']
props['Transactions']['x-aws-idp-source-page-types'] = ['TransactionsWorksheet']

print('Augmented Bank Statement class:')
print(yaml.safe_dump(
    {'x-aws-idp-page-types': bank_statement['x-aws-idp-page-types'],
     'properties': {k: {kk: vv for kk, vv in v.items() if kk == 'x-aws-idp-source-page-types'}
                    for k, v in props.items()}},
    sort_keys=False,
))

## 3. Run A — Control: full document, feature OFF

This is the baseline — exactly what `step3_extraction.ipynb` produces today. Every property in the schema is in the output (populated where the model found data, possibly null/empty otherwise). Notice that `extraction.missing_field_handling.enabled` is `False` (the default).

In [ ]:
config_a = copy.deepcopy(augmented)
config_a.setdefault('extraction', {})['missing_field_handling'] = {'enabled': False}

doc_a = Document.from_json(document.to_json())
service_a = extraction.ExtractionService(config=config_a)
doc_a = service_a.process_document_section(document=doc_a, section_id=section.section_id)

result_uri_a = doc_a.sections[0].extraction_result_uri
print(f'Extraction A complete: {result_uri_a}')

In [ ]:
def load_json_from_s3(uri: str) -> dict:
    s3 = boto3.client('s3')
    bucket, _, key = uri.replace('s3://', '').partition('/')
    return json.loads(s3.get_object(Bucket=bucket, Key=key)['Body'].read())

result_a = load_json_from_s3(result_uri_a)
print('inference_result keys:', sorted(result_a['inference_result'].keys()))
print('page_type_resolution present?', 'page_type_resolution' in result_a)
print('missing_fields_report present?', 'missing_fields_report' in result_a)
print()
print(json.dumps(result_a['inference_result'], indent=2, default=str)[:2000])

## 4. Run B — Simulate omission, feature ON

We mutate a fresh copy of the document to **remove every page that the regex identifies as a transactions worksheet** from `section.page_ids`. This mirrors what would happen if the submitter never sent the transactions pages — the rest of the pipeline wouldn't have OCR text for them, and `_load_page_texts` would skip them.

We also enable `extraction.missing_field_handling` so the post-processing pass marks fields whose source pages are absent.

In [ ]:
from idp_common import s3 as idp_s3
from idp_common.extraction.page_type_resolver import resolve_page_types

page_id_to_text = {}
for pid in section.page_ids:
    page = document.pages[pid]
    page_id_to_text[pid] = idp_s3.get_text_content(page.parsed_text_uri)

presence = resolve_page_types(bank_statement, page_id_to_text)
print('Detected page-types per page:')
for pid in sorted(section.page_ids, key=int):
    label = presence.page_id_to_page_type.get(pid, '(unmatched)')
    print(f'  page {pid}: {label}')

transactions_pages = [pid for pid, name in presence.page_id_to_page_type.items() if name == 'TransactionsWorksheet']
if not transactions_pages:
    raise RuntimeError(
        'Could not detect a TransactionsWorksheet page — the regex may not match this sample. '
        'Adjust the regex in cell 2 if needed.'
    )
# Drop ALL pages of this type so the page-type becomes genuinely absent. This
# sample has several TransactionsWorksheet pages, and the missing-field feature
# only fires when NONE of a field's source pages remain in the section.
pages_to_drop = set(transactions_pages)
print(f'\nWill drop pages {sorted(pages_to_drop, key=int)} (all TransactionsWorksheet) for Run B.')

In [ ]:
config_b = copy.deepcopy(augmented)
config_b.setdefault('extraction', {})['missing_field_handling'] = {
    'enabled': True,
    'representation': 'omit',  # also try 'null_with_metadata'
}

doc_b = Document.from_json(document.to_json())
section_b = doc_b.sections[0]
section_b.page_ids = [pid for pid in section_b.page_ids if pid not in pages_to_drop]
if section_b.attributes and 'page_indices' in section_b.attributes:
    # Recompute page_indices so the section's first page maps to index 0.
    section_b.attributes['page_indices'] = list(range(len(section_b.page_ids)))

print(f'Section pages after omission: {section_b.page_ids}')

service_b = extraction.ExtractionService(config=config_b)
doc_b = service_b.process_document_section(document=doc_b, section_id=section_b.section_id)

result_uri_b = doc_b.sections[0].extraction_result_uri
result_b = load_json_from_s3(result_uri_b)

print(f'\nExtraction B complete: {result_uri_b}')
print('page_type_resolution:', json.dumps(result_b.get('page_type_resolution'), indent=2))
print('missing_fields_report:', json.dumps(result_b.get('missing_fields_report'), indent=2))

## 5. Diff the two outputs

What changed between Run A (BLANK semantics) and Run B (MISSING semantics)? Fields whose source pages are still present should be unchanged in shape; fields whose source pages were dropped should be **omitted** from `inference_result` (under `representation: omit`) and listed in `missing_fields_report`.

In [ ]:
keys_a = set(result_a['inference_result'].keys())
keys_b = set(result_b['inference_result'].keys())

print(f'Keys in Run A inference_result : {sorted(keys_a)}')
print(f'Keys in Run B inference_result : {sorted(keys_b)}')
print()
print(f'Dropped in B (treated as MISSING): {sorted(keys_a - keys_b)}')
print(f'Only in B (unexpected)          : {sorted(keys_b - keys_a)}')

for field in sorted(keys_a & keys_b):
    val_a = result_a['inference_result'][field]
    val_b = result_b['inference_result'][field]
    if val_a != val_b:
        print(f'\n[changed] {field}:')
        print(f'  A: {json.dumps(val_a, default=str)[:200]}')
        print(f'  B: {json.dumps(val_b, default=str)[:200]}')

## 6. Try the `null_with_metadata` representation

Some downstream systems prefer a stable key set (every schema field always present) plus an explicit list of missing names. Re-running with `representation: 'null_with_metadata'` keeps the keys but sets them to `null` and emits a sibling `missing_fields` array.

In [ ]:
config_c = copy.deepcopy(augmented)
config_c.setdefault('extraction', {})['missing_field_handling'] = {
    'enabled': True,
    'representation': 'null_with_metadata',
}

doc_c = Document.from_json(document.to_json())
section_c = doc_c.sections[0]
section_c.page_ids = [pid for pid in section_c.page_ids if pid not in pages_to_drop]
if section_c.attributes and 'page_indices' in section_c.attributes:
    section_c.attributes['page_indices'] = list(range(len(section_c.page_ids)))

service_c = extraction.ExtractionService(config=config_c)
doc_c = service_c.process_document_section(document=doc_c, section_id=section_c.section_id)

result_c = load_json_from_s3(doc_c.sections[0].extraction_result_uri)

print('inference_result keys (should match Run A):', sorted(result_c['inference_result'].keys()))
print('missing_fields:', result_c.get('missing_fields'))
print()
for field in result_c.get('missing_fields', []):
    print(f'  {field} = {result_c["inference_result"][field]!r}  (now null + listed)')

## Summary

| Run | Pages | Feature | Behavior |
|-----|-------|---------|----------|
| A | All | OFF | Default: every schema field present in `inference_result`. No way to tell BLANK from MISSING. |
| B | Transactions page omitted | ON, `representation: omit` | Fields sourced from absent pages are **dropped** from `inference_result` and listed in `missing_fields_report`. |
| C | Transactions page omitted | ON, `representation: null_with_metadata` | Stable key set: keys remain present but null; absent fields listed in a `missing_fields` array. |

### When to enable this

This feature is only useful when:

- Your forms have legitimately optional sub-sections / supplemental pages, **and**
- Downstream systems need to distinguish "submitter didn't fill this in" from "submitter didn't include the page".

If neither applies, leave it off — the default behavior is unchanged.

### See also

- [docs/missing-page-handling.md](../../../docs/missing-page-handling.md) — full feature guide
- [config_library/unified/bank-statement-sample/config.yaml](../../../config_library/unified/bank-statement-sample/config.yaml) — annotated commented-out stanza you can uncomment in a deployed stack